In [1]:
import boto3
import time
bedrock = boto3.client('bedrock', region_name='us-east-2')
response = bedrock.list_foundation_models()
print(f"Available models: {len(response['modelSummaries'])}")
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-2')

Available models: 96


**Pipeline:** benchmark (50 questions, 10 domains) → every model answers (Converse API, tokens + latency recorded) → automatic metrics (BLEU, ROUGE, BERTScore) → LLM-as-judge scores 4 criteria → cost / latency analysis → HTML report with charts.

In [2]:
import os
import re
import io
import math
import json
import html
import base64
import webbrowser
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from botocore.exceptions import ClientError, BotoCoreError

bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-2')

# Models to compare. Prices are USD per 1M tokens (on-demand list prices, update them if they change).
MODELS = {
    "Haiku 4.5":  {"id": "us.anthropic.claude-haiku-4-5-20251001-v1:0", "input_price": 1.0, "output_price": 5.0,  "color": "#2a9d8f"},
    "Sonnet 4.6": {"id": "us.anthropic.claude-sonnet-4-6",              "input_price": 3.0, "output_price": 15.0, "color": "#e9a23b"},
    "Opus 4.6":   {"id": "us.anthropic.claude-opus-4-6-v1",             "input_price": 5.0, "output_price": 25.0, "color": "#7b5ea7"},
}
JUDGE_MODEL_ID = 'us.anthropic.claude-sonnet-4-6'
JUDGE_PRICES = (3.0, 15.0)  # (input, output) per 1M tokens
EMBED_MODEL_ID = 'amazon.titan-embed-text-v2:0'
EMBED_DIMENSIONS = 512

CACHE_FILE = Path("benchmark_cache.json")   # answers + judgments are cached here, so re-running costs nothing
REPORT_FILE = Path("evaluation_report.html")

**Task 1 - Benchmark.** 50 questions, 5 per domain, each with a short reference answer. Short references make BLEU / ROUGE / BERTScore meaningful; the LLM judge also gets the reference as ground truth.

In [3]:
# --- Task 1: benchmark with 50 questions from 10 domains ---
_RAW = {
    "Science": [
        ("What is photosynthesis?", "Photosynthesis is the process by which plants, algae and some bacteria use sunlight, water and carbon dioxide to produce glucose and oxygen."),
        ("Why is the sky blue?", "The sky is blue because molecules in the atmosphere scatter shorter blue wavelengths of sunlight more strongly than longer red wavelengths, an effect called Rayleigh scattering."),
        ("What does DNA stand for and what does it do?", "DNA stands for deoxyribonucleic acid; it stores the genetic instructions used for the growth, functioning and reproduction of living organisms."),
        ("What is Newton's second law of motion?", "Newton's second law states that force equals mass times acceleration (F = ma), so acceleration is proportional to the net force and inversely proportional to the mass."),
        ("What causes the seasons on Earth?", "Seasons are caused by the tilt of Earth's axis of about 23.5 degrees as it orbits the Sun, which changes the angle and duration of sunlight at different latitudes."),
    ],
    "Mathematics": [
        ("What is the Pythagorean theorem?", "In a right triangle the square of the hypotenuse equals the sum of the squares of the other two sides: a^2 + b^2 = c^2."),
        ("What is the derivative of x squared?", "The derivative of x squared with respect to x is 2x."),
        ("What is a prime number? Give the first five.", "A prime number is a natural number greater than 1 with no divisors other than 1 and itself; the first five are 2, 3, 5, 7 and 11."),
        ("If a train travels 120 km in 1.5 hours, what is its average speed?", "The average speed is 80 kilometers per hour."),
        ("What is the sum of the interior angles of a triangle?", "The interior angles of a triangle always add up to 180 degrees."),
    ],
    "History": [
        ("Who was the first president of the United States?", "George Washington was the first president of the United States, serving from 1789 to 1797."),
        ("In what year did World War II end?", "World War II ended in 1945, with Germany surrendering in May and Japan in September."),
        ("What was the Renaissance?", "The Renaissance was a cultural and intellectual movement that began in Italy in the 14th century and revived interest in classical art, science and learning in Europe."),
        ("Who built the pyramids of Giza?", "The pyramids of Giza were built by ancient Egyptians as tombs for the pharaohs Khufu, Khafre and Menkaure during the Fourth Dynasty."),
        ("What event started the French Revolution in 1789?", "The French Revolution began in 1789 with the storming of the Bastille in Paris on July 14, amid financial crisis and social inequality."),
    ],
    "Geography": [
        ("What is the capital of Australia?", "The capital of Australia is Canberra."),
        ("Which is the longest river in South America?", "The Amazon River is the longest river in South America."),
        ("Which is the largest ocean on Earth?", "The Pacific Ocean is the largest ocean on Earth."),
        ("What is the tallest mountain above sea level?", "Mount Everest, in the Himalayas on the border of Nepal and China, is the tallest mountain above sea level at about 8,849 meters."),
        ("Which is the most populous country in the world?", "India is the most populous country in the world, having surpassed China in 2023."),
    ],
    "Technology": [
        ("What is the difference between a process and a thread?", "A process is an independent program in execution with its own memory space, while a thread is a lightweight unit of execution within a process that shares the process's memory."),
        ("What does HTTP stand for?", "HTTP stands for Hypertext Transfer Protocol, the application protocol used to transfer web pages and data between clients and servers."),
        ("What is Big O notation?", "Big O notation describes the upper bound of an algorithm's growth rate in time or space as the input size increases."),
        ("What is a REST API?", "A REST API is a web interface that follows representational state transfer principles, using standard HTTP methods such as GET, POST, PUT and DELETE on resources identified by URLs."),
        ("What is the purpose of a database index?", "A database index is a data structure that speeds up data retrieval on a table at the cost of extra storage and slower writes."),
    ],
    "Literature & Arts": [
        ("Who wrote Don Quixote?", "Don Quixote was written by Miguel de Cervantes and published in two parts in 1605 and 1615."),
        ("Who painted the Mona Lisa?", "The Mona Lisa was painted by Leonardo da Vinci in the early 16th century."),
        ("Who wrote Romeo and Juliet?", "Romeo and Juliet was written by William Shakespeare; it is a tragedy about two young lovers from feuding families in Verona."),
        ("Who wrote One Hundred Years of Solitude?", "Gabriel Garcia Marquez, a Colombian author, wrote One Hundred Years of Solitude, a landmark of magical realism."),
        ("What is a haiku?", "A haiku is a short Japanese poem of three lines with a 5-7-5 syllable pattern, often focused on nature."),
    ],
    "Health": [
        ("What is the function of red blood cells?", "Red blood cells carry oxygen from the lungs to the body's tissues using hemoglobin and carry carbon dioxide back to the lungs."),
        ("What is the difference between a virus and a bacterium?", "Bacteria are single-celled living organisms that can reproduce on their own, whereas viruses are much smaller, need a host cell to replicate, and are not treated with antibiotics."),
        ("Why is sleep important?", "Sleep allows the body and brain to repair tissues, consolidate memory, regulate hormones and maintain immune function."),
        ("What does insulin do?", "Insulin is a hormone made by the pancreas that allows cells to take up glucose from the blood, lowering blood sugar levels."),
        ("What is a balanced diet?", "A balanced diet provides the right proportions of carbohydrates, proteins, fats, vitamins, minerals and water to meet the body's needs."),
    ],
    "Economics": [
        ("What is inflation?", "Inflation is the general rise in prices over time, which reduces the purchasing power of money."),
        ("What is the difference between revenue and profit?", "Revenue is the total income a company earns from sales, while profit is what remains after subtracting all expenses and costs from revenue."),
        ("What is supply and demand?", "Supply and demand is an economic model in which the price of a good is determined by the balance between how much sellers offer and how much buyers want."),
        ("What is a stock?", "A stock is a share of ownership in a company that entitles the holder to a portion of its assets and earnings."),
        ("What does GDP measure?", "Gross domestic product measures the total market value of all goods and services produced within a country in a given period."),
    ],
    "Language & Logic": [
        ("What is the plural of 'analysis'?", "The plural of analysis is analyses."),
        ("If all bloops are razzies and all razzies are lazzies, are all bloops lazzies?", "Yes, all bloops are lazzies, because the relationship is transitive."),
        ("Give three synonyms for 'happy'.", "Three synonyms for happy are joyful, cheerful and content."),
        ("Translate 'Good morning, how are you?' into Spanish.", "Buenos días, ¿cómo estás?"),
        ("What is the next number in the sequence 2, 4, 8, 16?", "The next number is 32, since each term is double the previous one."),
    ],
    "AWS & Cloud": [
        ("What is Amazon S3?", "Amazon S3 is an object storage service from AWS that offers scalable, durable and highly available storage for any amount of data."),
        ("What is AWS Lambda?", "AWS Lambda is a serverless compute service that runs code in response to events without requiring you to provision or manage servers, charging only for the compute time used."),
        ("What is an IAM role?", "An IAM role is an AWS identity with permission policies that can be assumed by users, applications or services to obtain temporary credentials."),
        ("What is an Availability Zone?", "An Availability Zone is one or more discrete data centers with redundant power, networking and connectivity within an AWS Region, isolated from failures in other zones."),
        ("What is Amazon Bedrock?", "Amazon Bedrock is a fully managed AWS service that provides access to foundation models from several providers through a single API for building generative AI applications."),
    ],
}
BENCHMARK = [{"id": i, "domain": domain, "question": q, "reference": ref}
             for i, (domain, q, ref) in enumerate(((d, q, r) for d, items in _RAW.items() for q, r in items), start=1)]

assert len(BENCHMARK) == 50
print(f"{len(BENCHMARK)} questions in {len(_RAW)} domains:", dict(Counter(b["domain"] for b in BENCHMARK)))
print("Example:", BENCHMARK[0])

50 questions in 10 domains: {'Science': 5, 'Mathematics': 5, 'History': 5, 'Geography': 5, 'Technology': 5, 'Literature & Arts': 5, 'Health': 5, 'Economics': 5, 'Language & Logic': 5, 'AWS & Cloud': 5}
Example: {'id': 1, 'domain': 'Science', 'question': 'What is photosynthesis?', 'reference': 'Photosynthesis is the process by which plants, algae and some bacteria use sunlight, water and carbon dioxide to produce glucose and oxygen.'}


**Calling the models.** One helper wraps the Converse API for candidates *and* the judge. It returns the text plus input/output tokens and latency, retries throttling with exponential backoff (as in exercise 4) and never raises. Answers and judgments are cached in `benchmark_cache.json`, so only missing or failed calls are repeated.

In [4]:
# --- Model calls: tokens + latency per request, retries, cache ---
_RETRYABLE = {'ThrottlingException', 'ServiceUnavailableException', 'ModelNotReadyException'}


def call_model(model_id, prompt, system=None, max_tokens=400, max_retries=4):
    """Single Converse call -> {'text', 'input_tokens', 'output_tokens', 'latency', 'error'}."""
    kwargs = {"modelId": model_id, "messages": [{"role": "user", "content": [{"text": prompt}]}],
              "inferenceConfig": {"maxTokens": max_tokens, "temperature": 0}}
    if system:
        kwargs["system"] = [{"text": system}]
    failure = {"text": "", "input_tokens": 0, "output_tokens": 0, "latency": None}
    for attempt in range(max_retries + 1):
        start = time.time()
        try:
            response = bedrock_runtime.converse(**kwargs)
        except ClientError as e:
            code = e.response['Error']['Code']
            if code in _RETRYABLE and attempt < max_retries:
                time.sleep(2 ** attempt)
                continue
            return {**failure, "error": f"{code}: {e.response['Error']['Message']}"}
        except BotoCoreError as e:
            return {**failure, "error": f"{type(e).__name__}: {e}"}
        usage = response["usage"]
        return {"text": " ".join(b["text"] for b in response["output"]["message"]["content"] if "text" in b).strip(),
                "input_tokens": usage["inputTokens"], "output_tokens": usage["outputTokens"],
                "latency": time.time() - start, "error": None}


def load_cache():
    if CACHE_FILE.exists():
        return json.loads(CACHE_FILE.read_text(encoding="utf-8"))
    return {"answers": {}, "judgments": {}}


def save_cache(cache):
    CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")


def run_parallel(jobs, work, cache_section, cache, workers=4, label=""):
    """Run `work(job) -> (key, result)` for every job in a thread pool and store results in the cache."""
    done = 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for key, result in pool.map(work, jobs):
            cache[cache_section][key] = result
            done += 1
            if done % 25 == 0 or done == len(jobs):
                print(f"  {label}: {done}/{len(jobs)}")
                save_cache(cache)  # partial progress survives a crash / expired credentials


ANSWER_SYSTEM = "Answer the question accurately and concisely in 1-3 sentences. No preamble and no markdown."


def collect_answers(cache, models=MODELS, questions=BENCHMARK):
    jobs = [(name, q) for name in models for q in questions
            if cache["answers"].get(f"{name}|{q['id']}", {}).get("error", "missing")]  # missing or failed
    print(f"Answers to generate: {len(jobs)} ({len(models) * len(questions) - len(jobs)} cached)")

    def work(job):
        name, q = job
        return f"{name}|{q['id']}", call_model(models[name]["id"], q["question"], ANSWER_SYSTEM)

    run_parallel(jobs, work, "answers", cache, label="answers")

**Task 2 - Automatic metrics** (answer vs. reference, all in the 0-1 range):
- **BLEU**: modified n-gram precision (1-4 grams, add-one smoothing for n>1) with a brevity penalty.
- **ROUGE-1 / ROUGE-2 / ROUGE-L**: F1 of unigram overlap, bigram overlap and longest common subsequence.
- **BERTScore**: greedy soft matching of embeddings, F1 of precision and recall. The original uses contextual *token* embeddings from a BERT model (needs PyTorch); Bedrock only exposes sentence embeddings, so this version matches *sentences* using the Titan embeddings from exercise 6. The scores are raw cosine similarities (no baseline rescaling), so they are higher than classic BERTScore values.

In [5]:
# --- Task 2: BLEU, ROUGE, BERTScore ---
def tokenize(text):
    return re.findall(r"\w+", text.lower())


def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def bleu(reference, hypothesis, max_n=4):
    ref, hyp = tokenize(reference), tokenize(hypothesis)
    if not hyp or not ref:
        return 0.0
    log_precision = 0.0
    for n in range(1, max_n + 1):
        overlap = sum((ngram_counts(hyp, n) & ngram_counts(ref, n)).values())
        total = max(len(hyp) - n + 1, 0)
        precision = overlap / total if n == 1 else (overlap + 1) / (total + 1)  # smoothing for higher orders
        if precision == 0:
            return 0.0
        log_precision += math.log(precision) / max_n
    brevity = 1.0 if len(hyp) > len(ref) else math.exp(1 - len(ref) / len(hyp))
    return brevity * math.exp(log_precision)


def _f1(overlap, ref_total, hyp_total):
    if not overlap:
        return 0.0
    precision, recall = overlap / hyp_total, overlap / ref_total
    return 2 * precision * recall / (precision + recall)


def rouge_n(reference, hypothesis, n):
    ref, hyp = ngram_counts(tokenize(reference), n), ngram_counts(tokenize(hypothesis), n)
    return _f1(sum((ref & hyp).values()), sum(ref.values()), sum(hyp.values()))


def rouge_l(reference, hypothesis):
    ref, hyp = tokenize(reference), tokenize(hypothesis)
    if not ref or not hyp:
        return 0.0
    row = [0] * (len(hyp) + 1)  # longest common subsequence, one row at a time
    for r in ref:
        prev, row = row, [0]
        for j, h in enumerate(hyp, start=1):
            row.append(prev[j - 1] + 1 if r == h else max(prev[j], row[j - 1]))
    return _f1(row[-1], len(ref), len(hyp))


# BERTScore-style metric on Titan embeddings (same embed() as exercise 6, normalized -> cosine = dot product)
def embed(text):
    body = json.dumps({"inputText": text, "dimensions": EMBED_DIMENSIONS, "normalize": True})
    response = bedrock_runtime.invoke_model(modelId=EMBED_MODEL_ID, body=body)
    return np.array(json.loads(response["body"].read())["embedding"], dtype=np.float32)


_embed_cache = {}


def embed_cached(text):
    if text not in _embed_cache:
        _embed_cache[text] = embed(text)
    return _embed_cache[text]


def split_sentences(text):
    return [s for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s] or [text]


def bertscore(reference, hypothesis):
    """Greedy matching: every candidate sentence is matched to its most similar reference sentence (precision)
    and vice versa (recall); returns the F1."""
    ref = np.vstack([embed_cached(s) for s in split_sentences(reference)])
    hyp = np.vstack([embed_cached(s) for s in split_sentences(hypothesis)])
    sim = hyp @ ref.T
    precision, recall = float(sim.max(axis=1).mean()), float(sim.max(axis=0).mean())
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


def automatic_metrics(reference, hypothesis):
    return {"bleu": bleu(reference, hypothesis), "rouge1": rouge_n(reference, hypothesis, 1),
            "rouge2": rouge_n(reference, hypothesis, 2), "rougeL": rouge_l(reference, hypothesis),
            "bertscore": bertscore(reference, hypothesis)}


# Sanity check: identical text scores 1, unrelated text scores close to 0
print(automatic_metrics("The capital of Australia is Canberra.", "The capital of Australia is Canberra."))
print(automatic_metrics("The capital of Australia is Canberra.", "Bananas are rich in potassium."))

{'bleu': 1.0, 'rouge1': 1.0, 'rouge2': 1.0, 'rougeL': 1.0, 'bertscore': 1.0000001192092896}
{'bleu': 0.0, 'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0, 'bertscore': 0.1503804326057434}


**LLM-as-judge.** A stronger model grades every answer against the reference on four criteria (1-5): *correctness*, *completeness*, *clarity* and *conciseness*. The judge runs at temperature 0, must reply with JSON, and is retried once if the JSON cannot be parsed. The overall score is the mean of the four criteria.

Note: the judge (Sonnet 4.6) is also one of the candidates, so it may favor its own style (self-preference bias). `JUDGE_MODEL_ID` can be changed to another model to check.

In [6]:
# --- LLM-as-judge ---
CRITERIA = ["correctness", "completeness", "clarity", "conciseness"]
JUDGE_SYSTEM = ("You are a strict, impartial evaluator of question-answering systems. "
                "Compare the candidate answer with the reference answer and score it. Reply with ONLY a JSON object.")


def judge_prompt(question, reference, answer):
    return (f"Question: {question}\n\nReference answer: {reference}\n\nCandidate answer: {answer}\n\n"
            "Score the candidate from 1 (very poor) to 5 (excellent) on each criterion:\n"
            "- correctness: facts agree with the reference and are not wrong\n"
            "- completeness: covers the key points of the reference\n"
            "- clarity: well written and easy to understand\n"
            "- conciseness: no unnecessary padding or repetition\n\n"
            'Reply with JSON only: {"correctness": n, "completeness": n, "clarity": n, "conciseness": n, "reasoning": "one short sentence"}')


def judge(question, reference, answer):
    error = None
    for _ in range(2):  # one retry if the reply is not valid JSON
        r = call_model(JUDGE_MODEL_ID, judge_prompt(question, reference, answer), JUDGE_SYSTEM, max_tokens=300)
        if r["error"]:
            error = r["error"]
            continue
        try:
            data = json.loads(re.search(r"\{.*\}", r["text"], re.S).group(0))
            scores = {c: int(data[c]) for c in CRITERIA}
            assert all(1 <= v <= 5 for v in scores.values())
            return {**scores, "reasoning": str(data.get("reasoning", "")), "input_tokens": r["input_tokens"],
                    "output_tokens": r["output_tokens"], "latency": r["latency"], "error": None}
        except Exception as e:
            error = f"Unparseable judge reply ({type(e).__name__}): {r['text'][:80]}"
    return {"error": error}


def collect_judgments(cache, models=MODELS, questions=BENCHMARK):
    by_id = {q["id"]: q for q in questions}
    jobs = []
    for name in models:
        for q in questions:
            key = f"{name}|{q['id']}"
            answer = cache["answers"].get(key, {})
            if answer.get("text") and cache["judgments"].get(key, {}).get("error", "missing"):
                jobs.append((key, q["id"]))
    print(f"Judgments to generate: {len(jobs)}")

    def work(job):
        key, qid = job
        q = by_id[qid]
        return key, judge(q["question"], q["reference"], cache["answers"][key]["text"])

    run_parallel(jobs, work, "judgments", cache, label="judgments")

**Run the evaluation.** Candidates answer all 50 questions, then the judge and the automatic metrics score every answer. Everything is stored in one tidy table (one row per model and question).

In [7]:
# --- Run: answers -> judge -> metrics -> results table ---
cache = load_cache()
collect_answers(cache)
save_cache(cache)
collect_judgments(cache)
save_cache(cache)


def cost_usd(input_tokens, output_tokens, input_price, output_price):
    return (input_tokens * input_price + output_tokens * output_price) / 1_000_000


def build_results(cache, models=MODELS, questions=BENCHMARK):
    rows = []
    for name, cfg in models.items():
        for q in questions:
            key = f"{name}|{q['id']}"
            a, j = cache["answers"].get(key), cache["judgments"].get(key, {})
            if not a or a["error"] or not a["text"]:
                continue
            rows.append({"model": name, "id": q["id"], "domain": q["domain"], "question": q["question"],
                         "reference": q["reference"], "answer": a["text"],
                         "latency": a["latency"], "input_tokens": a["input_tokens"], "output_tokens": a["output_tokens"],
                         "cost": cost_usd(a["input_tokens"], a["output_tokens"], cfg["input_price"], cfg["output_price"]),
                         "judge_reasoning": j.get("reasoning", ""),
                         **{c: j.get(c) for c in CRITERIA},
                         "judge_cost": cost_usd(j.get("input_tokens", 0), j.get("output_tokens", 0), *JUDGE_PRICES)})
    df = pd.DataFrame(rows)
    df["overall"] = df[CRITERIA].astype(float).mean(axis=1, skipna=False)

    # automatic metrics: embed all distinct sentences once, in parallel, then score locally
    sentences = {s for text in list(df["reference"]) + list(df["answer"]) for s in split_sentences(text)}
    with ThreadPoolExecutor(max_workers=8) as pool:
        list(pool.map(embed_cached, sorted(sentences)))
    metrics = pd.DataFrame([automatic_metrics(r, a) for r, a in zip(df["reference"], df["answer"])])
    return pd.concat([df, metrics], axis=1)


results = build_results(cache)
failed = len(MODELS) * len(BENCHMARK) - len(results)
print(f"{len(results)} scored answers" + (f" ({failed} failed, re-run the cell to retry them)" if failed else ""))
results.head()

Answers to generate: 0 (150 cached)
Judgments to generate: 0
150 scored answers


,model,id,domain,question,reference,answer,latency,input_tokens,output_tokens,cost,...,completeness,clarity,conciseness,judge_cost,overall,bleu,rouge1,rouge2,rougeL,bertscore
0,Haiku 4.5,1,Science,What is photosynthesis?,"Photosynthesis is the process by which plants,...","Photosynthesis is the process by which plants,...",1.221081,38,78,0.000428,...,5,5,4,0.001884,4.75,0.199310,0.439024,0.275000,0.365854,0.537215
1,Haiku 4.5,2,Science,Why is the sky blue?,The sky is blue because molecules in the atmos...,The sky is blue because of a phenomenon called...,2.999986,37,104,0.000557,...,5,5,4,0.001950,4.75,0.170148,0.419048,0.310680,0.304762,0.809515
2,Haiku 4.5,3,Science,What does DNA stand for and what does it do?,DNA stands for deoxyribonucleic acid; it store...,DNA stands for deoxyribonucleic acid. It is a ...,2.561689,42,45,0.000267,...,5,5,5,0.001719,5.00,0.233026,0.500000,0.320000,0.500000,0.731074
3,Haiku 4.5,4,Science,What is Newton's second law of motion?,Newton's second law states that force equals m...,Newton's second law of motion states that the ...,2.512726,40,70,0.000390,...,5,5,4,0.001785,4.75,0.237273,0.611765,0.385542,0.423529,0.784059
4,Haiku 4.5,5,Science,What causes the seasons on Earth?,Seasons are caused by the tilt of Earth's axis...,The seasons are caused by the tilt of Earth's ...,1.436072,38,102,0.000548,...,5,5,4,0.001971,4.75,0.163289,0.389381,0.270270,0.336283,0.791038


**Task 4 - Costs and latencies.** Cost = tokens x price per 1M tokens (candidate calls only; the judge cost is reported separately as evaluation overhead). Latency is the wall-clock time of each Converse call. Requests ran 4 at a time, so absolute values include some contention, but the comparison between models is fair. `quality_per_dollar` = judge overall score per USD spent on 1,000 questions.

In [8]:
# --- Task 4: cost and latency analysis per model ---
METRICS = ["bleu", "rouge1", "rouge2", "rougeL", "bertscore"]


def summarize(df, models=MODELS):
    g = df.groupby("model")
    s = pd.DataFrame({
        "judge_overall": g["overall"].mean(),
        **{c: g[c].mean() for c in CRITERIA},
        **{m: g[m].mean() for m in METRICS},
        "latency_mean": g["latency"].mean(), "latency_p50": g["latency"].median(), "latency_p95": g["latency"].quantile(0.95),
        "avg_input_tokens": g["input_tokens"].mean(), "avg_output_tokens": g["output_tokens"].mean(),
        "total_cost": g["cost"].sum(), "cost_per_question": g["cost"].mean(),
        "judge_cost": g["judge_cost"].sum(),
    }).reindex([m for m in models if m in g.groups])
    s["cost_per_1k_questions"] = s["cost_per_question"] * 1000
    s["quality_per_dollar"] = s["judge_overall"] / s["cost_per_1k_questions"]
    return s


summary = summarize(results)
pd.set_option("display.width", 200)
print(summary[["judge_overall", "bleu", "rouge1", "rougeL", "bertscore"]].round(3).to_string())
print()
print(summary[["latency_mean", "latency_p50", "latency_p95", "avg_output_tokens", "total_cost",
               "cost_per_1k_questions", "quality_per_dollar"]].round(4).to_string())
print(f"\nJudge overhead: ${summary['judge_cost'].sum():.4f} for {len(results)} judgments")

# How well do the automatic metrics agree with the judge? (Pearson correlation over all answers)
agreement = results[METRICS + ["overall"]].astype(float).corr()["overall"].drop("overall")
print("\nCorrelation with judge score:\n", agreement.round(3).to_string())

# Score by domain (judge overall)
by_domain = results.pivot_table(index="domain", columns="model", values="overall", aggfunc="mean")[list(summary.index)]
by_domain.round(2)

            judge_overall   bleu  rouge1  rougeL  bertscore
model                                                      
Haiku 4.5           4.820  0.294   0.536   0.465      0.801
Sonnet 4.6          4.815  0.324   0.566   0.519      0.807
Opus 4.6            4.825  0.308   0.549   0.505      0.795

            latency_mean  latency_p50  latency_p95  avg_output_tokens  total_cost  cost_per_1k_questions  quality_per_dollar
model                                                                                                                       
Haiku 4.5         1.1970       1.1021       2.1331              59.16      0.0168                 0.3358             14.3555
Sonnet 4.6        1.3357       1.3272       1.8045              52.90      0.0458                 0.9164              5.2544
Opus 4.6          1.7757       1.6734       2.6019              62.56      0.0884                 1.7688              2.7278

Judge overhead: $0.2601 for 150 judgments

Correlation with judge score:


model,Haiku 4.5,Sonnet 4.6,Opus 4.6
domain,,,
AWS & Cloud,4.70,4.75,4.75
Economics,4.75,4.80,4.75
Geography,4.80,4.70,4.85
Health,4.75,4.85,4.75
History,4.85,4.85,4.80
Language & Logic,4.95,4.90,5.00
Literature & Arts,4.85,4.75,4.80
Mathematics,4.95,4.95,4.90
Science,4.80,4.90,4.85


**Task 3 - HTML report.** A single self-contained file (`evaluation_report.html`, charts embedded as images): headline findings, comparison tables, charts for judge criteria, automatic metrics, domains, latency and cost vs. quality, plus every question with each model's answer and score.

In [ ]:
# --- Task 3: HTML report with comparative charts ---
def fig_to_base64(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=130, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return base64.b64encode(buffer.getvalue()).decode()


def _style(ax, title, ylabel=None):
    ax.set_title(title, loc="left", fontsize=12, fontweight="bold")
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)


def grouped_bars(summary, columns, labels, title, ylabel, ylim=None):
    fig, ax = plt.subplots(figsize=(8, 3.8))
    width = 0.8 / len(summary)
    x = np.arange(len(columns))
    for i, (name, row) in enumerate(summary.iterrows()):
        bars = ax.bar(x + i * width - 0.4 + width / 2, [row[c] for c in columns], width, label=name, color=MODELS[name]["color"])
        ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=2)
    ax.set_xticks(x, labels)
    if ylim:
        ax.set_ylim(*ylim)
    ax.legend(frameon=False, fontsize=8, ncols=len(summary))
    _style(ax, title, ylabel)
    return fig_to_base64(fig)


def domain_heatmap(by_domain):
    fig, ax = plt.subplots(figsize=(6.5, 4.6))
    image = ax.imshow(by_domain.values, cmap="YlGn", vmin=1, vmax=5, aspect="auto")
    ax.set_xticks(range(by_domain.shape[1]), by_domain.columns)
    ax.set_yticks(range(by_domain.shape[0]), by_domain.index)
    for (i, j), v in np.ndenumerate(by_domain.values):
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="black")
    ax.set_title("Judge score by domain (1-5)", loc="left", fontsize=12, fontweight="bold")
    ax.tick_params(length=0)
    fig.colorbar(image, ax=ax, fraction=0.04, pad=0.03)
    return fig_to_base64(fig)


def latency_boxplot(df, order):
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    box = ax.boxplot([df[df.model == m]["latency"] for m in order], tick_labels=order, patch_artist=True,
                     medianprops={"color": "black"}, showfliers=True, flierprops={"markersize": 3})
    for patch, m in zip(box["boxes"], order):
        patch.set_facecolor(MODELS[m]["color"])
    _style(ax, "Latency per request", "seconds")
    return fig_to_base64(fig)


def cost_quality_scatter(summary):
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for name, row in summary.iterrows():
        ax.scatter(row["cost_per_1k_questions"], row["judge_overall"], s=120, color=MODELS[name]["color"], zorder=3)
        ax.annotate(f"{name}\n${row['cost_per_1k_questions']:.2f} / 1k q", (row["cost_per_1k_questions"], row["judge_overall"]),
                    textcoords="offset points", xytext=(8, -4), fontsize=8)
    ax.set_xlim(0, summary["cost_per_1k_questions"].max() * 1.5)
    ax.set_ylim(max(1, summary["judge_overall"].min() - 0.4), min(5.05, summary["judge_overall"].max() + 0.4))
    ax.set_xlabel("cost per 1,000 questions (USD)")
    ax.grid(alpha=0.25)
    _style(ax, "Cost vs. quality", "judge overall score (1-5)")
    return fig_to_base64(fig)


def findings(summary):
    best = summary["judge_overall"].idxmax()
    cheapest = summary["cost_per_1k_questions"].idxmin()
    fastest = summary["latency_p50"].idxmin()
    value = summary["quality_per_dollar"].idxmax()
    return [f"<b>Best quality:</b> {best} (judge {summary.loc[best, 'judge_overall']:.2f}/5, BERTScore {summary.loc[best, 'bertscore']:.3f}).",
            f"<b>Cheapest:</b> {cheapest} (${summary.loc[cheapest, 'cost_per_1k_questions']:.2f} per 1,000 questions).",
            f"<b>Fastest:</b> {fastest} (median {summary.loc[fastest, 'latency_p50']:.2f}s, p95 {summary.loc[fastest, 'latency_p95']:.2f}s).",
            f"<b>Best value:</b> {value} ({summary.loc[value, 'quality_per_dollar']:.2f} judge points per USD per 1,000 questions)."]


REPORT_TEMPLATE = """<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Model Evaluation Report</title>
<style>
 body { font-family: -apple-system, "Segoe UI", Roboto, sans-serif; color: #1d2733; background: #f6f7f9; margin: 0; }
 main { max-width: 1000px; margin: 0 auto; padding: 24px 16px 60px; }
 h1 { margin-bottom: 4px; } h2 { margin-top: 36px; border-bottom: 1px solid #d9dee5; padding-bottom: 6px; }
 .sub { color: #5b6673; margin-top: 0; }
 .card { background: #fff; border: 1px solid #e1e5ea; border-radius: 8px; padding: 14px 18px; margin: 12px 0; }
 .grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(420px, 1fr)); gap: 12px; }
 img { max-width: 100%; height: auto; }
 table { border-collapse: collapse; width: 100%; font-size: 13px; background: #fff; }
 th, td { border: 1px solid #e1e5ea; padding: 5px 8px; text-align: right; vertical-align: top; }
 th { background: #eef1f5; } td.l, th.l { text-align: left; }
 .scroll { overflow-x: auto; } details { margin: 8px 0; } summary { cursor: pointer; font-weight: 600; }
 .ans { text-align: left; font-size: 12px; color: #3b4652; max-width: 260px; }
 ul { line-height: 1.7; }
</style></head><body><main>
<h1>Model Evaluation Report</h1>
<p class="sub">{{SUBTITLE}}</p>
<div class="card"><ul>{{FINDINGS}}</ul></div>

<h2>Summary</h2>
<div class="scroll">{{SUMMARY_TABLE}}</div>

<h2>Quality</h2>
<div class="grid"><div class="card"><img src="data:image/png;base64,{{CHART_CRITERIA}}"></div>
<div class="card"><img src="data:image/png;base64,{{CHART_METRICS}}"></div></div>
<div class="card"><p>Correlation of each automatic metric with the judge's overall score (Pearson, all answers):</p>{{AGREEMENT_TABLE}}</div>

<h2>By domain</h2>
<div class="grid"><div class="card"><img src="data:image/png;base64,{{CHART_DOMAIN}}"></div>
<div class="card scroll">{{DOMAIN_TABLE}}</div></div>

<h2>Latency and cost</h2>
<div class="grid"><div class="card"><img src="data:image/png;base64,{{CHART_LATENCY}}"></div>
<div class="card"><img src="data:image/png;base64,{{CHART_COST}}"></div></div>
<div class="scroll">{{COST_TABLE}}</div>

<h2>All questions</h2>
<details><summary>Show every question, reference and model answer ({{N_QUESTIONS}} questions)</summary>
<div class="scroll">{{QUESTIONS_TABLE}}</div></details>
</main></body></html>"""


def _table(df, fmt="{:.3f}", index_label=""):
    out = df.copy()
    out.index.name = index_label
    return out.to_html(float_format=lambda v: fmt.format(v), border=0, na_rep="-", escape=True)


def questions_table(df, order):
    header = "".join(f"<th class='l'>{html.escape(m)}</th>" for m in order)
    rows = []
    for qid, group in df.groupby("id"):
        first = group.iloc[0]
        cells = ""
        for m in order:
            r = group[group.model == m]
            if r.empty:
                cells += "<td class='ans'>(failed)</td>"
                continue
            r = r.iloc[0]
            score = "-" if pd.isna(r["overall"]) else f"{r['overall']:.2f}"
            cells += (f"<td class='ans'><b>{score}</b> &middot; {r['latency']:.1f}s<br>{html.escape(r['answer'])}"
                      f"<br><i>{html.escape(r['judge_reasoning'])}</i></td>")
        rows.append(f"<tr><td>{qid}</td><td class='l'>{html.escape(first['domain'])}</td>"
                    f"<td class='ans'>{html.escape(first['question'])}</td>"
                    f"<td class='ans'>{html.escape(first['reference'])}</td>{cells}</tr>")
    return (f"<table><tr><th>#</th><th class='l'>Domain</th><th class='l'>Question</th><th class='l'>Reference</th>{header}</tr>"
            + "".join(rows) + "</table>")


def generate_report(df, summary, by_domain, agreement, path=REPORT_FILE):
    order = list(summary.index)
    quality_cols = ["judge_overall"] + CRITERIA + METRICS
    cost_cols = ["latency_mean", "latency_p50", "latency_p95", "avg_input_tokens", "avg_output_tokens",
                 "total_cost", "cost_per_question", "cost_per_1k_questions", "quality_per_dollar", "judge_cost"]
    parts = {
        "SUBTITLE": f"{len(order)} models &middot; {df['id'].nunique()} questions &middot; {df['domain'].nunique()} domains &middot; "
                    f"judge: {html.escape(JUDGE_MODEL_ID)} &middot; generated {datetime.now():%Y-%m-%d %H:%M}",
        "FINDINGS": "".join(f"<li>{f}</li>" for f in findings(summary)),
        "SUMMARY_TABLE": _table(summary[quality_cols], "{:.3f}", "model"),
        "CHART_CRITERIA": grouped_bars(summary, CRITERIA, [c.title() for c in CRITERIA], "Judge scores by criterion", "score (1-5)", (0, 5.6)),
        "CHART_METRICS": grouped_bars(summary, METRICS, ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "BERTScore"],
                                      "Automatic metrics vs. reference", "score (0-1)", (0, 1.1)),
        "AGREEMENT_TABLE": _table(agreement.to_frame("correlation").T, "{:.2f}"),
        "CHART_DOMAIN": domain_heatmap(by_domain),
        "DOMAIN_TABLE": _table(by_domain, "{:.2f}", "domain"),
        "CHART_LATENCY": latency_boxplot(df, order),
        "CHART_COST": cost_quality_scatter(summary),
        "COST_TABLE": _table(summary[cost_cols], "{:.4f}", "model"),
        "N_QUESTIONS": str(df["id"].nunique()),
        "QUESTIONS_TABLE": questions_table(df, order),
    }
    page = REPORT_TEMPLATE
    for key, value in parts.items():
        page = page.replace("{{" + key + "}}", value)
    Path(path).write_text(page, encoding="utf-8")
    return Path(path).resolve()


report_path = generate_report(results, summary, by_domain, agreement)
print(f"Report saved to {report_path} ({report_path.stat().st_size / 1024:.0f} KB)")
for f in findings(summary):
    print("-", re.sub(r"<[^>]+>", "", f))
webbrowser.open(report_path.as_uri())

Report saved to C:\Users\MatiasSarmiento\Desktop\Onboarding\evaluation_report.html (292 KB)
- Best quality: Opus 4.6 (judge 4.83/5, BERTScore 0.795).
- Cheapest: Haiku 4.5 ($0.34 per 1,000 questions).
- Fastest: Haiku 4.5 (median 1.10s, p95 2.13s).
- Best value: Haiku 4.5 (14.36 judge points per USD per 1,000 questions).
